In [1]:
old_wd <- getwd()

setwd("../")

if (file.exists("renv/activate.R")) {
  source("renv/activate.R")
} else {
  stop("Could not find renv/activate.R in project root")
}

setwd(old_wd)

- The project is out-of-sync -- use `renv::status()` for details.


In [2]:
# Define the top-level directory where your subject folders are
root_dir <- "../ds006018_per_stimuli"

# Get all .vhrk files in any subfolder
all_vhrk_files <- list.files(
  path = root_dir,
  pattern = "*.csv$", # regex for your prefix
  recursive = TRUE,                                    # search subfolders
  full.names = TRUE                                    # give absolute/relative paths
)

# View the result
print(all_vhrk_files)

   [1] "../ds006018_per_stimuli/sub-001/task-auditoryoddball_Stimulus_S1.csv"             
   [2] "../ds006018_per_stimuli/sub-001/task-auditoryoddball_Stimulus_S180.csv"           
   [3] "../ds006018_per_stimuli/sub-001/task-auditoryoddball_Stimulus_S70.csv"            
   [4] "../ds006018_per_stimuli/sub-001/task-auditoryoddball_Stimulus_S80.csv"            
   [5] "../ds006018_per_stimuli/sub-001/task-flanker_Stimulus_S11.csv"                    
   [6] "../ds006018_per_stimuli/sub-001/task-flanker_Stimulus_S111.csv"                   
   [7] "../ds006018_per_stimuli/sub-001/task-flanker_Stimulus_S112.csv"                   
   [8] "../ds006018_per_stimuli/sub-001/task-flanker_Stimulus_S12.csv"                    
   [9] "../ds006018_per_stimuli/sub-001/task-flanker_Stimulus_S121.csv"                   
  [10] "../ds006018_per_stimuli/sub-001/task-flanker_Stimulus_S122.csv"                   
  [11] "../ds006018_per_stimuli/sub-001/task-flanker_Stimulus_S2.csv"                     

In [3]:
library(fda)
library(reshape2)

Loading required package: splines

Loading required package: fds

Loading required package: rainbow

Loading required package: MASS

Loading required package: pcaPP

Loading required package: RCurl

Loading required package: deSolve


Attaching package: ‘fda’


The following object is masked from ‘package:graphics’:

    matplot


The following object is masked from ‘package:datasets’:

    gait




In [4]:
#for (path in all_vhrk_files) {
    # 1. Load the data exported from Python
    #eeg_df <- read.csv(path)


    # 2. Format for FDA
    # Let's say we are looking at one channel (e.g., 'Cz')
    # We need a matrix where rows = time, cols = trials
    #eeg_matrix <- reshape2::acast(eeg_df, time ~ epoch, value.var = "F7")


    # 3. Define the Time Range and Basis
    #time_points <- as.numeric(rownames(eeg_matrix))
    #time_range <- range(time_points)

    # Create a B-spline basis (common for EEG)
    # nbasis should be enough to capture the peaks but small enough to smooth noise
    #eeg_basis <- create.bspline.basis(time_range, nbasis = 20)


    # 4. Smooth the data into a Functional Data (fd) object
    #eeg_fd <- smooth.basis(time_points, eeg_matrix, eeg_basis)$fd
    #eeg_fd

    # 5. Plot to verify
    #plot(eeg_fd, main="Functional EEG Curves (CP1)")

    # 6. Save the fd object for later use
    #saveRDS(eeg_fd, file = "eeg_functional_data.rds")
#}

In [5]:
# Define the root directories to make the code adaptable
input_root <- "ds006018_per_stimuli"
output_root <- "ds006018_functional"

for (path in all_vhrk_files) {
    # 1. Load the data
    eeg_df <- read.csv(path)

    # 2. Reshape and Process (FDA logic)
    eeg_matrix <- reshape2::acast(eeg_df, time ~ epoch, value.var = "F7")
    time_points <- as.numeric(rownames(eeg_matrix))
    time_range <- range(time_points)
    eeg_basis <- create.bspline.basis(time_range, nbasis = 20)
    eeg_fd <- smooth.basis(time_points, eeg_matrix, eeg_basis)$fd

    # 3. CONSTRUCT DYNAMIC PATHS
    # Replace the input root folder name with the output root folder name in the string
    new_path_full <- gsub(input_root, output_root, path)
    
    # Get the directory part (e.g., .../ds006018_functional/sub-016)
    target_dir <- dirname(new_path_full)
    
    # Get the filename without .csv (e.g., task-visualoddball_Stimulus_S51)
    file_name_clean <- tools::file_path_sans_ext(basename(path))
    
    # 4. CREATE FOLDER IF MISSING
    if (!dir.exists(target_dir)) {
        dir.create(target_dir, recursive = TRUE)
    }

    # 5. DEFINE FINAL RDS PATH AND SAVE
    final_rds_path <- file.path(target_dir, paste0(file_name_clean, ".rds"))
    
    saveRDS(eeg_fd, file = final_rds_path)
    
    # Feedback for the user
    message("Processed: ", file_name_clean)
    message("Saved to: ", final_rds_path)
    message("-----------------------------------")
}

Processed: task-auditoryoddball_Stimulus_S1

Saved to: ../ds006018_functional/sub-001/task-auditoryoddball_Stimulus_S1.rds

-----------------------------------

Processed: task-auditoryoddball_Stimulus_S180

Saved to: ../ds006018_functional/sub-001/task-auditoryoddball_Stimulus_S180.rds

-----------------------------------

Processed: task-auditoryoddball_Stimulus_S70

Saved to: ../ds006018_functional/sub-001/task-auditoryoddball_Stimulus_S70.rds

-----------------------------------

Processed: task-auditoryoddball_Stimulus_S80

Saved to: ../ds006018_functional/sub-001/task-auditoryoddball_Stimulus_S80.rds

-----------------------------------

Processed: task-flanker_Stimulus_S11

Saved to: ../ds006018_functional/sub-001/task-flanker_Stimulus_S11.rds

-----------------------------------

Processed: task-flanker_Stimulus_S111

Saved to: ../ds006018_functional/sub-001/task-flanker_Stimulus_S111.rds

-----------------------------------

Processed: task-flanker_Stimulus_S112

Saved to: ../

In [6]:
# ==============================================================================
# Fix filenames containing ':' (invalid on Windows)
# ==============================================================================

output_dir <- "../ds006018_functional"

bad_files <- list.files(output_dir, pattern = ":", recursive = TRUE, full.names = TRUE)

if (length(bad_files) > 0) {
  cat("Found", length(bad_files), "files with ':' in the name (invalid on Windows):\n")
  for (f in bad_files) {
    new_name <- gsub(":", "_", f)
    file.rename(f, new_name)
    cat("  Renamed:", basename(f), "->", basename(new_name), "\n")
  }
  cat("Done. All colons replaced with underscores.\n")
} else {
  cat("No problematic filenames found.\n")
}

Found 2 files with ':' in the name (invalid on Windows):
  Renamed: task-visualoddball_NewSegment_LostSamples:264.rds -> task-visualoddball_NewSegment_LostSamples_264.rds 
  Renamed: task-visualsearch_NewSegment_LostSamples:1072.rds -> task-visualsearch_NewSegment_LostSamples_1072.rds 
Done. All colons replaced with underscores.
